<img src="https://upload.wikimedia.org/wikipedia/commons/3/35/Uba_fiuba_ingenieria_logo.png" width="300" align="center">



# **Analisis de Series de Tiempo II**

# **Clase 4, Global Forecasting Models**

Silenciamos los mensajes informativos de TensorFlow

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

Importamos las librerias necesarias

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

Desactivamos los avisos de retracing, estos son esperables al crear 14 modelos en un bucle

In [ ]:
tf.get_logger().setLevel("ERROR")

Fijamos las semillas para reproducibilidad

In [ ]:
np.random.seed(42)
keras.utils.set_random_seed(42)

Descargamos el panel mensual de desempleo por industria (BLS, 2000-2010)

In [ ]:
url = "https://raw.githubusercontent.com/vega/vega-datasets/main/data/unemployment-across-industries.json"
df = pd.read_json(url)

In [ ]:
df.head()

,series,year,month,count,rate,date
0,Government,2000,1,430,2.1,2000-01-01 08:00:00+00:00
1,Government,2000,2,409,2.0,2000-02-01 08:00:00+00:00
2,Government,2000,3,311,1.5,2000-03-01 08:00:00+00:00
3,Government,2000,4,269,1.3,2000-04-01 08:00:00+00:00
4,Government,2000,5,370,1.9,2000-05-01 07:00:00+00:00


Pivoteamos: cada fila es un mes, cada columna una industria

In [ ]:
panel = df.pivot_table(index="date", columns="series", values="count")

In [ ]:
panel.head()

series,Agriculture,Business services,Construction,Education and Health,Finance,Government,Information,Leisure and hospitality,Manufacturing,Mining and Extraction,Other,Self-employed,Transportation and Utilities,Wholesale and Retail Trade
date,,,,,,,,,,,,,,
2000-01-01 08:00:00+00:00,154.0,655.0,745.0,353.0,228.0,430.0,125.0,782.0,734.0,19.0,274.0,239.0,236.0,1000.0
2000-02-01 08:00:00+00:00,173.0,587.0,812.0,349.0,240.0,409.0,112.0,779.0,694.0,25.0,232.0,262.0,223.0,1023.0
2000-03-01 08:00:00+00:00,152.0,623.0,669.0,381.0,226.0,311.0,140.0,789.0,739.0,17.0,247.0,213.0,192.0,983.0
2000-04-01 08:00:00+00:00,135.0,517.0,447.0,329.0,197.0,269.0,95.0,658.0,736.0,20.0,240.0,218.0,191.0,793.0
2000-05-01 07:00:00+00:00,73.0,561.0,397.0,423.0,195.0,370.0,131.0,675.0,685.0,27.0,254.0,206.0,190.0,821.0


Guardamos la lista de industrias que forman el panel

In [ ]:
industrias = list(panel.columns)
print(f"Panel: {panel.shape[0]} meses x {len(industrias)} industrias")

Panel: 122 meses x 14 industrias


Definimos la longitud de la ventana:

* 12 meses captura un ciclo estacional completo

In [ ]:
L = 12

Definimos el tamaño del test:

* los ultimos 24 meses DE CADA SERIE

Este periodo contiene la crisis de 2008, un quiebre estructural nunca visto en train

In [ ]:
N_TEST = 24

Definimos el tamaño del estado oculto, identico en el enfoque local y el global

In [ ]:
UNIDADES = 16

Creamos diccionarios para guardar los datos de cada industria por separado

In [ ]:
X_tr, y_tr, X_te, y_te, escalas = {}, {}, {}, {}, {}

Recorremos industria por industria construyendo su dataset supervisado

In [ ]:
for ind in industrias:

    # Extraemos la serie de la industria como array de numpy
    v = panel[ind].values.astype("float32")

    # Calculamos media y desvio usando solo el tramo de entrenamiento de esta serie
    mu = v[:-N_TEST].mean()
    sd = v[:-N_TEST].std()

    # Guardamos las estadisticas para poder desestandarizar el error despues
    escalas[ind] = (mu, sd)

    # Estandarizamos la serie
    z = (v - mu) / sd

    # Construimos las ventanas deslizantes de esta industria
    Xs, ys = [], []
    for t in range(L, len(z)):

        # Guardamos los 12 meses previos como input
        Xs.append(z[t - L:t].reshape(-1, 1))

        # Guardamos el mes siguiente como target
        ys.append(z[t])

    # Convertimos a arrays con la forma que espera Keras
    Xs = np.array(Xs, dtype="float32")
    ys = np.array(ys, dtype="float32").reshape(-1, 1)

    # Realizamos el split temporal dentro de esta serie
    # Error clasico a evitar: agrupar todas las series y despues cortar por indice,
    # porque eso deja series enteras en test y otras enteras en train
    X_tr[ind], X_te[ind] = Xs[:-N_TEST], Xs[-N_TEST:]
    y_tr[ind], y_te[ind] = ys[:-N_TEST], ys[-N_TEST:]

Mostramos cuantas ventanas de entrenamiento aporta cada industria

In [ ]:
print(f"Ventanas de entrenamiento por industria: {len(X_tr[industrias[0]])}")

Ventanas de entrenamiento por industria: 86


### **A) Baseline: Seasonal Naive**

El seasonal naive predice que el mes de hoy sera igual al mismo mes del año pasado


Como la ventana tiene 12 meses, ese valor es la primera posicion de la ventana

In [ ]:
mae_snaive = {}
for ind in industrias:

    # Tomamos el primer elemento de cada ventana de test como prediccion
    pred = X_te[ind][:, 0, 0:1]

    # Calculamos el MAE y lo llevamos a la escala original
    mae_snaive[ind] = np.abs(pred - y_te[ind]).mean() * escalas[ind][1]

### **B) Local: Un Modelo Por Industria**

Creamos el diccionario donde guardaremos el error de cada modelo local

In [ ]:
mae_local = {}

Contamos el total de parametros que hay que mantener con el enfoque local

In [ ]:
params_local = 0

Entrenamos 14 modelos locales

In [ ]:
for ind in industrias:

    # Definimos la entrada: 12 pasos temporales con 1 variable
    entrada = keras.Input(shape=(L, 1))

    # Aplicamos la capa recurrente
    h = layers.LSTM(UNIDADES)(entrada)

    # Proyectamos a la prediccion escalar
    salida = layers.Dense(1)(h)

    # Ensamblamos el modelo exclusivo de esta industria
    modelo = keras.Model(entrada, salida)

    # Compilamos con la misma configuracion que usaremos en el global
    modelo.compile(optimizer=keras.optimizers.Adam(0.01), loss="mse")

    # Acumulamos los parametros de este modelo al total del enfoque local
    params_local += modelo.count_params()

    # Entrenamos solo con las ventanas de esta industria
    modelo.fit(X_tr[ind], y_tr[ind], epochs=100, batch_size=16, verbose=0)

    # Predecimos sobre el test de esta industria
    pred = modelo.predict(X_te[ind], verbose=0)

    # Calculamos el MAE en la escala original
    mae_local[ind] = np.abs(pred - y_te[ind]).mean() * escalas[ind][1]

    # Liberamos memoria del grafo antes de construir el siguiente modelo
    keras.backend.clear_session()

### **C) Global: Un Modelo Para las 14 Industrias**

Apilamos las ventanas de entrenamiento de todas las industrias en un solo pool


Esta es la unica diferencia conceptual con el enfoque local: **el pooling**

In [ ]:
X_pool = np.concatenate([X_tr[ind] for ind in industrias], axis=0)
y_pool = np.concatenate([y_tr[ind] for ind in industrias], axis=0)

Mostramos el tamaño del pool: 14 veces mas datos para el mismo modelo

In [ ]:
print(f"\nPool global: {X_pool.shape[0]} ventanas")


Pool global: 1204 ventanas


Definimos el modelo global con exactamente la misma arquitectura que los locales

In [ ]:
entrada_g = keras.Input(shape=(L, 1), name="ventana")
h_g = layers.LSTM(UNIDADES)(entrada_g)
salida_g = layers.Dense(1, name="prediccion")(h_g)
modelo_global = keras.Model(entrada_g, salida_g)

Compilamos con la misma configuracion

In [ ]:
modelo_global.compile(optimizer=keras.optimizers.Adam(0.01), loss="mse")

Guardamos la cantidad de parametros del unico modelo global

In [ ]:
params_global = modelo_global.count_params()

Entrenamos sobre el pool completo

In [ ]:
print("Entrenando 1 modelo global...")
modelo_global.fit(X_pool, y_pool, epochs=100, batch_size=64, verbose=0)

Entrenando 1 modelo global...


Evaluamos el modelo global sobre el test de cada industria por separado


El promedio esconde exactamente lo que queremos ver: la variacion por serie

In [ ]:
mae_global = {}
for ind in industrias:
    pred = modelo_global.predict(X_te[ind], verbose=0)
    mae_global[ind] = np.abs(pred - y_te[ind]).mean() * escalas[ind][1]

### **D) Comparativa**

Recorremos las industrias ordenadas por el error del modelo global

In [ ]:
print(f"{'Industria':<30}{'S-Naive':>11}{'Local':>11}{'Global':>11}{'Gana':>9}")
for ind in sorted(industrias, key=lambda s: mae_global[s]):
    # Determinamos que enfoque gano en esta industria
    ganador = "global" if mae_global[ind] < mae_local[ind] else "LOCAL"
    print(f"{ind:<30}{mae_snaive[ind]:>11.0f}{mae_local[ind]:>11.0f}"
          f"{mae_global[ind]:>11.0f}{ganador:>9}")

Industria                         S-Naive      Local     Global     Gana
Mining and Extraction                  38         42         29   global
Information                            90         48         42   global
Agriculture                            65         62         48   global
Other                                 130         81         69   global
Government                            177        149        121   global
Finance                               156        208        132   global
Transportation and Utilities          150        143        135   global
Self-employed                         141        185        139   global
Business services                     406        252        219   global
Education and Health                  290        313        227   global
Wholesale and Retail Trade            462        353        258   global
Leisure and hospitality               323        421        268   global
Manufacturing                         595        38

Comparamos el costo de mantenimiento de cada enfoque

In [ ]:
print(f"\nParametros totales a mantener:")
print(f"  Enfoque local:  {params_local:>8,} ({len(industrias)} modelos)")
print(f"  Enfoque global: {params_global:>8,} (1 modelo)")
print(f"  Reduccion: {(1 - params_global/params_local)*100:.0f}%")


Parametros totales a mantener:
  Enfoque local:    16,366 (14 modelos)
  Enfoque global:    1,169 (1 modelo)
  Reduccion: 93%


Interpretamos el resultado sin convertirlo en una regla universal

1.- El global gana con 93% menos parametros: no es solo mas barato, es mejor.

2.- Varios modelos locales ni siquiera le ganan al seasonal naive.